In [ ]:
import pandas as pd
import requests
import os
from datetime import datetime
print ("HDB Project ")

# Github repo kdhivyadharshni-ship-it


In [ ]:
RAW_PATH = "../data/raw/"

os.makedirs(RAW_PATH, exist_ok=True)

print("RAW folder ready:", RAW_PATH)

In [ ]:
# GEt the reponse from API to get metadata of all 5 datasets 

import requests

collection_id = 189

url = f"https://api-production.data.gov.sg/v2/public/api/collections/{collection_id}/metadata"
# url =f"https://data.gov.sg/collections/189/view"

response = requests.get(url)

metadata = response.json()
# print(meta_data.keys())
print(metadata)
print(json.dumps(metadata, indent=2)[:2000])

In [ ]:
# Print datasets
child_datasets = metadata["data"]["collectionMetadata"]["childDatasets"]
print(f'{child_datasets}')

In [ ]:
# Use the reteieved dataset id from the api call to get data

# dataset_id = "d_8b84c4ee58e3cfc0ece0d773c8ca6abc"
import time
import requests
from datetime import datetime

RAW_PATH = "raw_data"
os.makedirs(RAW_PATH, exist_ok=True)
all_dataframes = []

url = "https://data.gov.sg/api/action/datastore_search"
for i,dataset in enumerate(child_datasets) :
    params = {
    "resource_id": dataset,
    "limit": 10,
    "offset": 0
    }
    time.sleep(5)
    response = requests.get(url,params=params)
    Data = response.json()
    print(Data.keys())
    records = Data["result"]["records"]
    # print (f"{dataset} : {Data["result"]["records"]}")
    df = pd.DataFrame(records)
    df["dataset_id"] = dataset_id
    # df.head()

    file_path = f"{RAW_PATH}/hdb_file_{i+1}.csv"
    df.to_csv(file_path,index=False)
    print(f"Saved: {file_path} | Rows: {len(df)}")
    all_dataframes.append(df)




# df.head()

In [ ]:
# temporay
dataset = "d_ea9ed51da2787afaf8e51f827c304208"
import time
import requests
from datetime import datetime

df_master = pd.read_csv("C:/Users/kdhiv/Downloads/ResaleFlatPricesBasedonApprovalDate19901999.csv")
# print(Data.keys())
# records = Data["result"]["records"]
#     # print (f"{dataset} : {Data["result"]["records"]}")
# df = pd.DataFrame(records)
df_master




In [ ]:
df_master = pd.concat(all_dataframes, ignore_index = True)
df_master.head()

In [ ]:
from datetime import datetime
# df_master.info()
df_master.describe(include='all')
df_master.isnull().sum()

# Datatype validation
df_master["resale_price"] = pd.to_numeric(df_master["resale_price"],errors="coerce")
df_master["lease_commence_date"] = pd.to_numeric(df_master["lease_commence_date"],errors="coerce")
df_master["floor_area_sqm"] = pd.to_numeric(df_master["floor_area_sqm"],errors="coerce")


In [ ]:
print("Shape:", df_master.shape)
# df_master.info()
missing = df_master.isnull().sum()
missing
df_master.info()


In [ ]:
# BUSINESS RULE 
# missing = df_master.isnull().sum()
# missing_percent = (missing / len(df_master)) * 100

# missing_df = pd.DataFrame({
#     "missing_count": missing,
#     "missing_percent": missing_percent
# })

# missing_df

In [ ]:
# Storey classification using range -analytical

In [ ]:
# Validation Rule 
# Unique values 
for col in df.columns:
    print(f"\nColumn: {col}")
    print("Unique count:", df[col].nunique())
    print(df[col].unique())

In [ ]:
# Frequency Distribution
df_master["town"].value_counts()
df_master["flat_type"].value_counts()

In [ ]:
# Data profiling was performed using descriptive statistics and exploratory analysis.
# It was observed that several numerical fields such as resale_price and floor_area_sqm were stored as strings and required type conversion.
# Missing values were identified in the remaining_lease column (~60%), which aligns with differences in dataset versions.
# Categorical fields such as town, flat_type, and flat_model were analyzed to derive valid domain values.

In [ ]:
# !pip install pandera
import pandera.pandas as pa
from pandera import Column, DataFrameSchema ,Check
# Define validation schema
schema = DataFrameSchema({

    "month": Column(
        str,
        Check.str_matches(r"\d{4}-\d{2}")  # YYYY-MM format
    ),

    "town": Column(
        str,
        Check.isin(df_master["town"].unique())
    ),

    "flat_type": Column(
        str,
        Check.isin(df_master["flat_type"].unique())
    ),

    "flat_model": Column(
        str,
        Check.isin(df_master["flat_model"].unique())
    ),

    "storey_range": Column(
        str,
        Check.str_matches(r"\d{2} TO \d{2}")
    ),

    "floor_area_sqm": Column(
        float,
        Check.gt(0)
    ),

    "resale_price": Column(
        int,
        Check.gt(0)
    ),

    "lease_commence_date": Column(
        int,
        Check.ge(1960)  # assumption based on HDB history
    )
})

In [242]:
# Pandera used to validate schema 
try:
    validated_df = schema.validate(df_master, lazy=True)
    failed_validated_df = pd.DataFrame()
except pa.errors.SchemaError as e:
    print(e)
    validated_df = df_master.drop(index=e.failure_cases["index"].unique())

    #  Failed rows
    failed_validated_df = df_master.loc[e.failure_cases["index"].unique()]
    
failed_validated_df

""


In [ ]:
# Recompute Remaining Lease
from datetime import datetime

current_year = datetime.now().year
current_month = datetime.now().month

# convert to numeric
# df_master["lease_commence_date"] = pd.to_numeric(
#     df_master["lease_commence_date"], errors="coerce"
# )

# calculate total months
df_master["remaining_months"] = (
    (99 * 12)
    - ((current_year - df_master["lease_commence_date"]) * 12 + current_month)
)

# avoid negatives
df_master["remaining_months"] = df_master["remaining_months"].clip(lower=0)

# convert to years + months
years = (df_master["remaining_months"] // 12).astype(int)
months = (df_master["remaining_months"] % 12).astype(int)

df_master["remaining_lease_recomputed"] = (
    years.astype(str) + " years " + months.astype(str) + " months"
)
# uncomment later 
# df_master[["remaining_lease", "remaining_lease_recomputed"]].head()

In [247]:
# Create a unique identifier
# Hashing can be used to secure this code
# Step 3: Block → 3 digit code
import re
import pandas as pd

df = df.drop(columns=["avg_price_prefix"], errors="ignore")

# ✅ Step 0: Ensure correct types
df["resale_price"] = pd.to_numeric(df["resale_price"], errors="coerce")
df["month"] = df["month"].astype(str)
df["town"] = df["town"].astype(str)
df["flat_type"] = df["flat_type"].astype(str)

# Step 3: Block → 3 digit code
def format_block(block):
    digits = re.sub(r"\D", "", str(block))
    return digits[:3].zfill(3)

df["block_code"] = df["block"].apply(format_block)

# Step 4: Month → last 2 digits
df["month_code"] = df["month"].str[-2:]

# Step 5: Town initial
df["town_initial"] = df["town"].str[0]

# Step 6: Compute average resale price
grouped = df.groupby(["month", "town", "flat_type"])["resale_price"].mean().reset_index()

# Step 7: Take first 2 digits safely
grouped["avg_price_prefix"] = (
    grouped["resale_price"]
    .fillna(0)
    .astype(int)
    .astype(str)
    .str.zfill(2)
    .str[:2]
)

# Step 8: Merge back
df = df.merge(
    grouped[["month", "town", "flat_type", "avg_price_prefix"]],
    on=["month", "town", "flat_type"],
    how="left"
)

#  Debug check (VERY IMPORTANT)
print("Missing avg_price_prefix:", df["avg_price_prefix"].isnull().sum())

# Fill missing if any
df["avg_price_prefix"] = df["avg_price_prefix"].fillna("00")

# Step 9: Create Resale Identifier
df["resale_identifier"] = (
    "S"
    + df["block_code"]
    + df["avg_price_prefix"]
    + df["month_code"]
    + df["town_initial"]
)

# Preview
df_Transformed = df

Missing avg_price_prefix: 0


In [ ]:
import hashlib

def hash_id(x):
    # safety: handle nulls just in case
    if pd.isna(x):
        return None
    return hashlib.sha256(x.encode()).hexdigest()

df["hashed_identifier"] = df["resale_identifier"].apply(hash_id)
df["hashed_identifier"]


In [ ]:
# If there are any duplicate records, take the higher price and discard the lower price one. 

df['resale_price'] = pd.to_numeric(df['resale_price'],errors = "coerce")
# df = df.drop(columns =["avg_price_prefix_x","avg_price_prefix_y"],errors = "ignore")
# create composite field 
key_column = [col for col in df.columns if col != "resale_price"]  
key_column
df_sorted = df.sort_values (by = "resale_price" ,ascending = False ) 

df_cleaned = df_sorted.drop_duplicates(subset= key_column, keep= "first")
df_cleaned

# Records with lower price
# Step 5: Get failed (lower price duplicates)
df_failed_duplicates = df_sorted[~df_sorted.index.isin(df_cleaned.index)]

# Reset index
df_cleaned = df_cleaned.reset_index(drop=True)
df_failed_duplicates = df_failed_duplicates.reset_index(drop=True)

print("Cleaned rows:", len(df_cleaned))
print("Failed duplicates:", len(df_failed_duplicates))

In [249]:
import os

folders = ["raw", "cleaned", "transformed", "failed", "hashed"]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created ")

df_master.to_csv("raw/hdb_raw.csv", index=False)

df_cleaned.to_csv("cleaned/hdb_cleaned.csv", index=False)

# Failed :

df_failed = pd.concat([
    df_failed_duplicates,
    failed_validated_df   # if you created this
], ignore_index=True)

df_failed.to_csv("failed/hdb_failed.csv", index=False)

df_hashed = df_cleaned.copy()

df_hashed["hashed_identifier"] = df_hashed["resale_identifier"].apply(hash_id)

df_hashed.to_csv("hashed/hdb_hashed.csv", index=False)

df_transformed.to_csv("transformed/hdb_transformed.csv", index=False)

Folders created 


NameError: name 'df_transformed' is not defined